In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/mt_dataset_pref4feat16.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4675 entries, 0 to 4674
Columns: 164 entries, x1_linspace to x4_f15
dtypes: float64(160), int64(4)
memory usage: 5.8 MB


In [46]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# -----------------------------
# 1) Колонки
# -----------------------------
feat_names = [
    #"linspace",
    "linspace_session_id",
    "linspace_within_session",
    #"brightness",
    #"time_reaction",
    "omission_percent",
    #"symbol_m", "symbol_c", "symbol_s", "symbol_y", "symbol_f", "symbol_j",
    "location_0", "location_1", "location_2", "location_3",
    "participant_J", "participant_F", "participant_Y",
    #"correct_symbol_m", "correct_symbol_c", "correct_symbol_s", "correct_symbol_y", "correct_symbol_f", "correct_symbol_j",
]
latent_dim = 16
feat_names_fs = [f"f{i}" for i in range(latent_dim)]
feat_names.extend(feat_names_fs)

F_DIM = len(feat_names)

def idxs(feat_names, names):
    return [feat_names.index(n) for n in names]

base_names = (
    ["location_0", "location_1", "location_2", "location_3"] +
    [f"f{i}" for i in range(16)]
)

aux_names = ["omission_percent", "linspace_within_session", "linspace_session_id"]
participant_names = ["participant_J", "participant_F", "participant_Y"]

base_idx = idxs(feat_names, base_names)
aux_idx = idxs(feat_names, aux_names)
participant_idx = idxs(feat_names, participant_names)


def cols(prefix: str):
    return [f"{prefix}_{f}" for f in feat_names]

x1_cols = cols("x1")
x2_cols = cols("x2")
x3_cols = cols("x3")
x4_cols = cols("x4")

def prepare_arrays(df: pd.DataFrame):
    needed = x1_cols + x2_cols + x3_cols + x4_cols
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    X1 = df[x1_cols].to_numpy(dtype=np.float32)
    X2 = df[x2_cols].to_numpy(dtype=np.float32)
    X3 = df[x3_cols].to_numpy(dtype=np.float32)
    X4 = df[x4_cols].to_numpy(dtype=np.float32)
    return X1, X2, X3, X4

# -----------------------------
# 2) Dataset: возвращает четверку
# -----------------------------
class QuadDataset(Dataset):
    def __init__(self, X1, X2, X3, X4):
        self.X1 = torch.from_numpy(X1).float()
        self.X2 = torch.from_numpy(X2).float()
        self.X3 = torch.from_numpy(X3).float()
        self.X4 = torch.from_numpy(X4).float()

    def __len__(self):
        return self.X1.shape[0]

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.X4[idx]

class ParticipantVectorHead(nn.Module):
    """
    Хранит 3 обучаемых вектора w_J, w_F, w_Y (как матрица W: [3, H]).
    Выбирает нужный w по participant one-hot и считает dot(h, w).
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        # W[p] = w_p, shape (3, H)
        self.W = nn.Parameter(torch.randn(3, hidden_dim) * 0.02)

    def forward(self, h: torch.Tensor, p_onehot: torch.Tensor):
        """
        h: (B, H)
        p_onehot: (B, 3) one-hot [J,F,Y]
        returns:
          s: (B,)   score = sum_i h_i * w_i(participant)
          w_sel: (B, H) выбранные вектора (для диагностики, можно не возвращать)
        """
        # idx: (B,)
        idx = p_onehot.argmax(dim=1)

        # w_sel: (B, H)
        w_sel = self.W[idx]

        # dot product -> (B,)
        s = (h * w_sel).sum(dim=1)
        return s, w_sel


class RewardMoEVector(nn.Module):
    """
    Общая FFN -> h, затем экспертная "векторная голова" выбирается по participant.
    """
    def __init__(
        self,
        in_dim: int,
        participant_idx: tuple[int, int, int],
        hidden_dim: int = 64,
        trunk_hidden=(128, 64),   # скрытые размеры внутри trunk (можно пусто)
        dropout: float = 0.2,
        layernorm: bool = True,
        return_debug: bool = False,
    ):
        super().__init__()
        self.participant_idx = participant_idx
        self.return_debug = return_debug

        layers = []
        d = in_dim

        # trunk: несколько слоёв
        for h in trunk_hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            if layernorm:
                layers += [nn.LayerNorm(h)]
            d = h

        # последний слой trunk выводит вектор h размера hidden_dim
        layers += [nn.Linear(d, hidden_dim)]
        if layernorm:
            layers += [nn.LayerNorm(hidden_dim)]

        self.trunk = nn.Sequential(*layers)

        # "эксперты" = 3 обучаемых вектора
        self.head = ParticipantVectorHead(hidden_dim)

    def forward(self, x: torch.Tensor):
        pj, pf, py = self.participant_idx
        p = x[:, [pj, pf, py]]               # (B,3)

        h = self.trunk(x)                    # (B,H)
        s, w_sel = self.head(h, p)           # (B,), (B,H)

        if self.return_debug:
            return s, {"h": h, "w": w_sel, "p": p}
        return s

class ParticipantCosineHead(nn.Module):
    """
    3 экспертных вектора W[p] (3,H).
    Score = cosine(h, W[p]).
    """
    def __init__(self, hidden_dim: int, eps: float = 1e-8, learn_scale: bool = True):
        super().__init__()
        self.W = nn.Parameter(torch.randn(3, hidden_dim) * 0.02)
        self.eps = eps

        # Опционально: обучаемый масштаб, чтобы скор мог "растягиваться"
        # (часто полезно для listwise/pairwise)
        self.scale = nn.Parameter(torch.tensor(1.0)) if learn_scale else None

    def forward(self, h: torch.Tensor, p_onehot: torch.Tensor):
        idx = p_onehot.argmax(dim=1)       # (B,)
        w = self.W[idx]                    # (B,H)

        # cosine similarity
        h_norm = h.norm(dim=1).clamp_min(self.eps)       # (B,)
        w_norm = w.norm(dim=1).clamp_min(self.eps)       # (B,)
        cos = (h * w).sum(dim=1) / (h_norm * w_norm)     # (B,)

        if self.scale is not None:
            cos = cos * self.scale

        return cos

class RewardMoECosine(nn.Module):
    def __init__(
        self,
        in_dim: int,
        participant_idx: tuple[int, int, int],
        hidden_dim: int = 64,
        trunk_hidden=(128, 64),
        dropout: float = 0.2,
        layernorm: bool = True,
        eps: float = 1e-8,
        learn_scale: bool = True,
    ):
        super().__init__()
        self.participant_idx = participant_idx

        layers = []
        d = in_dim
        for h in trunk_hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            if layernorm:
                layers += [nn.LayerNorm(h)]
            d = h

        # выдаём вектор h
        layers += [nn.Linear(d, hidden_dim)]
        if layernorm:
            layers += [nn.LayerNorm(hidden_dim)]
        self.trunk = nn.Sequential(*layers)

        self.head = ParticipantCosineHead(hidden_dim, eps=eps, learn_scale=learn_scale)

    def forward(self, x: torch.Tensor):
        pj, pf, py = self.participant_idx
        p = x[:, [pj, pf, py]]      # (B,3)

        h = self.trunk(x)           # (B,H)
        s = self.head(h, p)         # (B,)
        return s

def build_mlp(in_dim: int, out_dim: int, hidden=(64, 64), dropout=0.2):
    layers = []
    d = in_dim
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
        d = h
    layers += [nn.Linear(d, out_dim)]
    return nn.Sequential(*layers)


class ExpertMoEHard(nn.Module):
    def __init__(self, aux_dim: int, out_dim: int, hidden=(64, 64), dropout=0.2):
        super().__init__()
        self.out_dim = out_dim
        self.experts = nn.ModuleList([
            build_mlp(aux_dim, out_dim, hidden=hidden, dropout=dropout),  # J
            build_mlp(aux_dim, out_dim, hidden=hidden, dropout=dropout),  # F
            build_mlp(aux_dim, out_dim, hidden=hidden, dropout=dropout),  # Y
        ])

    def forward(self, aux: torch.Tensor, p_onehot: torch.Tensor) -> torch.Tensor:
        idx = p_onehot.argmax(dim=1)  # (B,)
        w = torch.empty(aux.size(0), self.out_dim, device=aux.device, dtype=aux.dtype)

        for k in (0, 1, 2):
            mask = (idx == k)
            if mask.any():
                w[mask] = self.experts[k](aux[mask])
        return w

class ExpertMoEHardSharedTrunk(nn.Module):
    """
    Shared trunk: aux -> z, then hard-routed heads: z -> w
    """
    def __init__(
        self,
        aux_dim: int,
        out_dim: int,
        *,
        trunk_hidden=(64,),
        head_hidden=(64,),
        dropout=0.2,
    ):
        super().__init__()
        self.out_dim = out_dim

        # общий trunk: aux -> z_dim
        z_dim = trunk_hidden[-1] if len(trunk_hidden) > 0 else aux_dim
        if len(trunk_hidden) > 0:
            self.trunk = build_mlp(aux_dim, z_dim, hidden=trunk_hidden[:-1], dropout=dropout) \
                if len(trunk_hidden) > 1 else nn.Sequential(
                    nn.Linear(aux_dim, z_dim), nn.ReLU(), nn.Dropout(dropout)
                )
        else:
            self.trunk = nn.Identity()
            z_dim = aux_dim

        # 3 головы: z -> out_dim
        self.heads = nn.ModuleList([
            build_mlp(z_dim, out_dim, hidden=head_hidden, dropout=dropout),  # J
            build_mlp(z_dim, out_dim, hidden=head_hidden, dropout=dropout),  # F
            build_mlp(z_dim, out_dim, hidden=head_hidden, dropout=dropout),  # Y
        ])

    def forward(self, aux: torch.Tensor, p_onehot: torch.Tensor) -> torch.Tensor:
        z = self.trunk(aux)                 # (B, z_dim)
        idx = p_onehot.argmax(dim=1)        # (B,)

        w = torch.empty(z.size(0), self.out_dim, device=z.device, dtype=z.dtype)

        for k in (0, 1, 2):
            mask = (idx == k)
            if mask.any():
                w[mask] = self.heads[k](z[mask])
        return w

class ParticipantRotate2HH(nn.Module):
    """
    2×Householder для каждого participant.
    Храним два вектора (U1, U2) размера (3, D).
    Применяем:
        w1 = HH(w, u1)
        w2 = HH(w1, u2)
    """
    def __init__(self, dim: int, eps: float = 1e-8, init_scale: float = 0.005):
        super().__init__()
        self.eps = eps
        self.U1 = nn.Parameter(torch.randn(3, dim) * init_scale)
        self.U2 = nn.Parameter(torch.randn(3, dim) * init_scale)

    def _hh(self, w: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
        # w,u: (B,D)
        uw = (u * w).sum(dim=1, keepdim=True)                      # (B,1)
        uu = (u * u).sum(dim=1, keepdim=True).clamp_min(self.eps)  # (B,1)
        return w - 2.0 * u * (uw / uu)

    def forward(self, w: torch.Tensor, p_onehot: torch.Tensor) -> torch.Tensor:
        idx = p_onehot.argmax(dim=1)  # (B,)
        u1 = self.U1[idx]             # (B,D)
        u2 = self.U2[idx]             # (B,D)
        w = self._hh(w, u1)
        w = self._hh(w, u2)
        return w

class RewardMoECosineSplit(nn.Module):
    """
    Trunk(base)->h, MoE(aux, participant)->w, затем w' = rotate(w, participant),
    score = cosine(h, w').
    """
    def __init__(
        self,
        in_dim_full: int,
        base_idx: list[int],
        aux_idx: list[int],
        participant_idx: list[int],
        *,
        embed_dim: int = 64,
        trunk_hidden=(128, 64),
        expert_hidden=(64, 64),
        dropout=0.2,
        learn_scale=True,
        layernorm_trunk=True,
        layernorm_w=False,
        layernorm_h=False,
        rotate_init_scale=0.005,   # насколько “сильным” будет поворот на старте
    ):
        super().__init__()
        self.base_idx = base_idx
        self.aux_idx = aux_idx
        self.participant_idx = participant_idx

        # trunk: base -> h
        trunk_layers = []
        d = len(base_idx)
        for h in trunk_hidden:
            trunk_layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            if layernorm_trunk:
                trunk_layers += [nn.LayerNorm(h)]
            d = h
        trunk_layers += [nn.Linear(d, embed_dim)]
        if layernorm_trunk:
            trunk_layers += [nn.LayerNorm(embed_dim)]
        self.trunk = nn.Sequential(*trunk_layers)

        # experts: aux -> w
        self.moe = ExpertMoEHard(aux_dim=len(aux_idx), out_dim=embed_dim, hidden=expert_hidden, dropout=dropout)

        #self.moe = ExpertMoEHardSharedTrunk(
        #    aux_dim=len(aux_idx),
        #    out_dim=embed_dim,
        #    trunk_hidden=(512,),
        #    head_hidden=(256,192,128,64),
        #    dropout=dropout,
        #)

        self.norm_h = nn.LayerNorm(embed_dim) if layernorm_h else None
        self.norm_w = nn.LayerNorm(embed_dim) if layernorm_w else None

        # NEW: participant-specific rotation of w
        self.rotate_w = ParticipantRotate2HH(embed_dim, init_scale=rotate_init_scale)

        self.scale = nn.Parameter(torch.tensor(1.0)) if learn_scale else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = x[:, self.base_idx]
        aux  = x[:, self.aux_idx]
        p    = x[:, self.participant_idx]  # (B,3)

        h = self.trunk(base)   # (B,D)
        w = self.moe(aux, p)   # (B,D)

        # (опционально) нормировки -> потом normalize
        if self.norm_h is not None:
            h = self.norm_h(h)
        if self.norm_w is not None:
            w = self.norm_w(w)

        # NEW: повернуть/отразить w по участнику
        w = self.rotate_w(w, p)

        # cosine на unit-векторах = dot
        h = F.normalize(h, dim=1)
        w = F.normalize(w, dim=1)
        s = (h * w).sum(dim=1)

        if self.scale is not None:
            s = s * self.scale

        return s

def loss_listwise_top1(s1, s2, s3, s4, temperature=1.0, l2_scores=9e-3):
    """
    Listwise: -log softmax winner (x1).
    L = -s1 + logsumexp([s1,s2,s3,s4])
    """
    scores0 = torch.stack([s1, s2, s3, s4], dim=1)
    scores = torch.stack([s1, s2, s3, s4], dim=1) / temperature  # (B,4)
    # target winner index = 0
    loss = F.cross_entropy(scores, torch.zeros(scores.size(0), dtype=torch.long, device=scores.device), label_smoothing=0.05)
    # small penalty to keep scores bounded
    loss = loss + l2_scores * (scores0**2).mean()
    return loss

def eval_metrics(model, loader, device, temperature=1.0):
    """
    Оценка на loader для задач top-1 из 4:
      - loss (listwise или pairwise)
      - top1_acc_soft: top-1 с корректной обработкой ties (частичный кредит 1/k)
      - top1_acc_hard: обычный argmax top-1 (может завышать при ties)
      - tie_rate: доля примеров, где максимум не уникален
      - mean_margin: mean(s1 - max(s2,s3,s4))
      - pwin_*: статистики по P(x1 wins) = softmax(scores/T)[0]
    """
    model.eval()

    total_loss = 0.0
    total = 0

    # top-1 метрики
    top1_soft_sum = 0.0
    top1_hard_correct = 0

    # tie diagnostics
    tie_count = 0

    # margin
    margin_sum = 0.0

    # P(win) stats
    p_list = []

    for x1, x2, x3, x4 in loader:
        x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

        s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)
        scores = torch.stack([s1, s2, s3, s4], dim=1)  # (B,4)

        # loss
        logits = scores / temperature
        loss = F.cross_entropy(logits, torch.zeros(logits.size(0), dtype=torch.long, device=logits.device))

        bs = scores.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

        # ----- hard top1 (может врать при ties)
        pred = scores.argmax(dim=1)
        top1_hard_correct += int((pred == 0).sum().item())

        # ----- soft top1 (корректно учитывает ties)
        maxv = scores.max(dim=1, keepdim=True).values          # (B,1)
        is_max = (scores == maxv).float()                      # (B,4)
        k = is_max.sum(dim=1)                                  # (B,) сколько максимумов
        # частичный кредит: если x1 среди максимумов, то 1/k
        top1_soft_sum += float((is_max[:, 0] / k).sum().item())

        # tie rate: где максимум не уникален (k>1)
        tie_count += int((k > 1).sum().item())

        # margin: s1 - max(s2,s3,s4)
        best_loser = torch.max(scores[:, 1:], dim=1).values
        margin = (scores[:, 0] - best_loser)
        margin_sum += float(margin.mean().item()) * bs

        # P(win) согласованно с температурой
        pwin = torch.softmax(scores / temperature, dim=1)[:, 0]
        p_list.append(pwin.detach().cpu().numpy())

    p = np.concatenate(p_list) if len(p_list) else np.array([np.nan], dtype=np.float32)

    def p_stats(arr):
        return {
            "mean": float(np.mean(arr)),
            "std":  float(np.std(arr)),
            "q05":  float(np.quantile(arr, 0.05)),
            "q50":  float(np.quantile(arr, 0.50)),
            "q95":  float(np.quantile(arr, 0.95)),
            "min":  float(np.min(arr)),
            "max":  float(np.max(arr)),
        }

    return {
        "loss": total_loss / max(total, 1),
        "top1_acc_soft": top1_soft_sum / max(total, 1),
        "top1_acc_hard": top1_hard_correct / max(total, 1),
        "tie_rate": tie_count / max(total, 1),
        "mean_margin": margin_sum / max(total, 1),
        "pwin": p_stats(p),
    }
def _filter_test_by_participant_onehot(
    df_test: pd.DataFrame,
    participant: str,          # "J" / "F" / "Y"
    *,
    require_consistent=True,   # проверять, что participant одинаков для x1..x4
) -> pd.DataFrame:
    if participant not in ("J", "F", "Y"):
        raise ValueError("participant must be one of: 'J', 'F', 'Y'")

    c1 = f"x1_participant_{participant}"
    c2 = f"x2_participant_{participant}"
    c3 = f"x3_participant_{participant}"
    c4 = f"x4_participant_{participant}"

    missing = [c for c in (c1, c2, c3, c4) if c not in df_test.columns]
    if missing:
        raise ValueError(f"Missing participant one-hot columns: {missing}")

    sub = df_test[df_test[c1] == 1.0].copy()

    if require_consistent and len(sub) > 0:
        # проверяем, что по этой же букве в x2..x4 тоже 1.0
        ok = (sub[c2] == 1.0) & (sub[c3] == 1.0) & (sub[c4] == 1.0)
        # если есть несогласованные строки — выкинем их, чтобы метрика была честной
        sub = sub[ok].copy()

    return sub


@torch.no_grad()
def eval_test_for_participant(
    df_test: pd.DataFrame,
    participant: str,  # "J"/"F"/"Y"
    model,
    scaler,
    device,
    *,
    temperature=1.0,
    batch_size=256,
    require_consistent=True,
):
    """
    Считает метрики на тестовой выборке только для строк конкретного participant.

    Возвращает dict:
      - n: сколько строк в подвыборке
      - metrics: результат eval_metrics (или None если n=0)
    """
    sub = _filter_test_by_participant_onehot(
        df_test, participant, require_consistent=require_consistent
    )
    n = len(sub)
    if n == 0:
        return {"participant": participant, "n": 0, "metrics": None}

    # arrays из df
    X1, X2, X3, X4 = prepare_arrays(sub)

    # scaler (fit на train!) применяем одинаково
    def tr(x):
        return scaler.transform(x).astype(np.float32)

    X1, X2, X3, X4 = tr(X1), tr(X2), tr(X3), tr(X4)

    loader = DataLoader(QuadDataset(X1, X2, X3, X4), batch_size=batch_size, shuffle=False)
    metrics = eval_metrics(model, loader, device, temperature=temperature)

    return {"participant": participant, "n": n, "metrics": metrics}


def eval_test_all_participants(
    df_test: pd.DataFrame,
    model,
    scaler,
    device,
    *,
    participants=("J", "F", "Y"),
    temperature=1.0,
    batch_size=256,
    require_consistent=True,
) -> pd.DataFrame:
    """
    Возвращает таблицу метрик по каждому participant.
    """
    rows = []
    for p in participants:
        out = eval_test_for_participant(
            df_test, p, model, scaler, device,
            temperature=temperature,
            batch_size=batch_size,
            require_consistent=require_consistent,
        )
        n = out["n"]
        m = out["metrics"]
        if n == 0 or m is None:
            rows.append({
                "participant": p,
                "n": n,
                "loss": np.nan,
                "top1_soft": np.nan,
                "top1_hard": np.nan,
                "tie_rate": np.nan,
                "mean_margin": np.nan,
                "pwin_mean": np.nan,
                "pwin_q05": np.nan,
                "pwin_q95": np.nan,
            })
            continue

        rows.append({
            "participant": p,
            "n": n,
            "loss": m["loss"],
            "top1_soft": m["top1_acc_soft"],
            "top1_hard": m["top1_acc_hard"],
            "tie_rate": m["tie_rate"],
            "mean_margin": m["mean_margin"],
            "pwin_mean": m["pwin"]["mean"],
            "pwin_q05": m["pwin"]["q05"],
            "pwin_q95": m["pwin"]["q95"],
        })

    return pd.DataFrame(rows).sort_values("n", ascending=False)

def train_ranknet_quads(
    df: pd.DataFrame,
    *,
    temperature=1.0,

    test_size=0.2,      # доля TEST от всего df
    val_size=0.25,       # доля VAL от оставшегося после test

    random_state=0,
    batch_size=64,
    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    hidden=(64, 32),
    dropout=0.1,
    device=None,
    monitor="val_top1_acc", #val_loss or val_top1_acc
    min_delta=0.0,
    restore_best=True,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- Split по индексам, чтобы легко вернуть df_test ---
    idx = np.arange(len(df))

    idx_tmp, idx_te = train_test_split(
        idx, test_size=test_size, random_state=random_state, shuffle=True
    )
    idx_tr, idx_va = train_test_split(
        idx_tmp, test_size=val_size, random_state=random_state, shuffle=True
    )

    df_tr = df.iloc[idx_tr].reset_index(drop=True)
    df_va = df.iloc[idx_va].reset_index(drop=True)
    df_te = df.iloc[idx_te].reset_index(drop=True)

    # --- arrays ---
    X1_tr, X2_tr, X3_tr, X4_tr = prepare_arrays(df_tr)
    X1_va, X2_va, X3_va, X4_va = prepare_arrays(df_va)
    X1_te, X2_te, X3_te, X4_te = prepare_arrays(df_te)

    # --- scaler fit только на TRAIN кандидатах ---
    scaler = StandardScaler()
    scaler.fit(np.vstack([X1_tr, X2_tr, X3_tr, X4_tr]))

    def tr(x): return scaler.transform(x).astype(np.float32)

    X1_tr, X2_tr, X3_tr, X4_tr = tr(X1_tr), tr(X2_tr), tr(X3_tr), tr(X4_tr)
    X1_va, X2_va, X3_va, X4_va = tr(X1_va), tr(X2_va), tr(X3_va), tr(X4_va)
    X1_te, X2_te, X3_te, X4_te = tr(X1_te), tr(X2_te), tr(X3_te), tr(X4_te)

    train_loader = DataLoader(QuadDataset(X1_tr, X2_tr, X3_tr, X4_tr),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(QuadDataset(X1_va, X2_va, X3_va, X4_va),
                              batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(QuadDataset(X1_te, X2_te, X3_te, X4_te),
                              batch_size=batch_size, shuffle=False)

    #model = RewardMoEVector(
    #    in_dim=F_DIM,
    #    participant_idx=participant_idx,
    #    hidden_dim=32,
    #    trunk_hidden=hidden,
    #    dropout=dropout,
    #    layernorm=False,
    #    return_debug=False,
    #).to(device)

    #model = RewardMoECosine(
    #    in_dim=F_DIM,
    #    participant_idx=participant_idx,
    #    hidden_dim=32,
    #    trunk_hidden=hidden,
    #    dropout=0.2,
    #    layernorm=True,
    #    learn_scale=True,   # рекомендую оставить True
    #).to(device)

    model = RewardMoECosineSplit(
        in_dim_full=F_DIM,
        base_idx=base_idx,
        aux_idx=aux_idx,
        participant_idx=participant_idx,
        embed_dim=64,
        trunk_hidden=hidden,
        expert_hidden=(128, 64),
        dropout=dropout,
        learn_scale=True,
    ).to(device)

    # AdamW предпочтительнее, но оставлю как у тебя Adam, если хочешь:
    #optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    optim = torch.optim.AdamW([
    {"params": model.trunk.parameters(), "lr": lr},
    {"params": model.moe.parameters(),   "lr": 9e-4},
    {"params": model.rotate_w.parameters(),   "lr": 9e-4},
    {"params": [model.scale],           "lr": 1e-4} if model.scale is not None else {},
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optim, T_max=epochs, eta_min=1e-6
    )

    # --- best tracking ---
    if monitor == "val_loss":
        mode = "min"
        best_score = float("inf")
    elif monitor == "val_top1_acc":
        mode = "max"
        best_score = -float("inf")
    else:
        raise ValueError("monitor must be 'val_loss' or 'val_top1_acc'")

    best_epoch = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        for x1, x2, x3, x4 in train_loader:
            x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

            s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)

            loss = loss_listwise_top1(s1, s2, s3, s4, temperature=temperature)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        scheduler.step()

        trm = eval_metrics(model, train_loader, device, temperature=temperature)
        vam = eval_metrics(model, val_loader, device, temperature=temperature)

        print(
            f"epoch {epoch:03d} | "
            f"train loss {trm['loss']:.4f} top1_soft {trm['top1_acc_soft']:.3f} "
            f"tie {trm['tie_rate']:.3f} margin {trm['mean_margin']:.3f} | "
            f"val loss {vam['loss']:.4f} top1_soft {vam['top1_acc_soft']:.3f} "
            f"tie {vam['tie_rate']:.3f} margin {vam['mean_margin']:.3f} | "
            f"val Pwin mean {vam['pwin']['mean']:.3f} q05 {vam['pwin']['q05']:.3f} q95 {vam['pwin']['q95']:.3f}"
        )

        report = eval_test_all_participants(
            df_va, model, scaler, device,
            temperature=temperature
        )
        print(report)

        current = vam["loss"] if monitor == "val_loss" else vam["top1_acc_soft"]
        improved = (current < best_score - min_delta) if mode == "min" else (current > best_score + min_delta)

        if improved:
            best_score = current
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if restore_best and best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} with {monitor}={best_score:.4f}")

    # --- финальная оценка на test ---
    test_metrics = eval_metrics(model, test_loader, device, temperature=temperature)
    print(
        f"[TEST] loss {test_metrics['loss']:.4f} top1_soft {test_metrics['top1_acc_soft']:.3f} "
        f"tie {test_metrics['tie_rate']:.3f} margin {test_metrics['mean_margin']:.3f} | "
        f"Pwin mean {test_metrics['pwin']['mean']:.3f} q05 {test_metrics['pwin']['q05']:.3f} q95 {test_metrics['pwin']['q95']:.3f}"
    )

    info = {
        "best_epoch": best_epoch,
        "best_score": best_score,
        "monitor": monitor,
        "test_metrics": test_metrics,
        "sizes": {"train": len(df_tr), "val": len(df_va), "test": len(df_te)},
        "indices": {"train": idx_tr, "val": idx_va, "test": idx_te},
    }

    splits = {"df_train": df_tr, "df_val": df_va, "df_test": df_te}
    return model, scaler, device, info, splits

In [47]:
temp = 1.1
model, scaler, device, monitor_dict, splits = train_ranknet_quads(
    df,
    temperature=temp,
    epochs=400,
    lr=2e-3,
    weight_decay=1e-3,
    hidden=(128, 96),
    dropout=0.4,
    batch_size=512,
)

epoch 001 | train loss 1.1806 top1_soft 0.541 tie 0.000 margin -0.017 | val loss 1.1940 top1_soft 0.535 tie 0.000 margin -0.034 | val Pwin mean 0.313 q05 0.173 q95 0.408
  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  358  1.214373   0.539106   0.539106       0.0    -0.070500   
2           Y  307  1.097019   0.602606   0.602606       0.0     0.042749   
1           F  270  1.277150   0.451852   0.451852       0.0    -0.072973   

   pwin_mean  pwin_q05  pwin_q95  
0   0.309441  0.156554  0.407033  
2   0.341163  0.194666  0.419439  
1   0.284657  0.193003  0.377887  
epoch 002 | train loss 1.1078 top1_soft 0.546 tie 0.000 margin -0.095 | val loss 1.1136 top1_soft 0.535 tie 0.000 margin -0.097 | val Pwin mean 0.361 q05 0.112 q95 0.534
  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  358  1.162088   0.539106   0.539106       0.0    -0.148735   
2           Y  307  0.955938   0.625407   0.625407       

In [48]:
df_test = splits["df_test"]

report = eval_test_all_participants(
    df_test, model, scaler, device,
    temperature=temp
)
print(report)

  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  347  1.167400   0.530259   0.530259       0.0    -0.078277   
2           Y  305  0.935928   0.629508   0.629508       0.0     0.222903   
1           F  283  1.168336   0.530035   0.530035       0.0    -0.079993   

   pwin_mean  pwin_q05  pwin_q95  
0   0.353700  0.113897  0.600632  
2   0.424860  0.143136  0.642512  
1   0.343283  0.114813  0.534687  


In [ ]:
import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler


def train_one_fold_quads(
    *,
    random_state = 0,
    df_train_fold: pd.DataFrame,
    df_val_fold: pd.DataFrame,
    df_test_holdout: pd.DataFrame,
    hidden=(64, 32),
    dropout=0.1,
    temperature=1.0,
    batch_size=64,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-3,
    device=None,
    monitor="val_loss",     # "val_loss" или "val_top1_acc"
    min_delta=0.0,
    restore_best=True,
    fold_id,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- arrays ---
    X1_tr, X2_tr, X3_tr, X4_tr = prepare_arrays(df_train_fold)
    X1_va, X2_va, X3_va, X4_va = prepare_arrays(df_val_fold)
    X1_te, X2_te, X3_te, X4_te = prepare_arrays(df_test_holdout)

    # --- scaler только на TRAIN фолда ---
    scaler = StandardScaler()
    scaler.fit(np.vstack([X1_tr, X2_tr, X3_tr, X4_tr]))

    def tr(x): return scaler.transform(x).astype(np.float32)

    X1_tr, X2_tr, X3_tr, X4_tr = tr(X1_tr), tr(X2_tr), tr(X3_tr), tr(X4_tr)
    X1_va, X2_va, X3_va, X4_va = tr(X1_va), tr(X2_va), tr(X3_va), tr(X4_va)
    X1_te, X2_te, X3_te, X4_te = tr(X1_te), tr(X2_te), tr(X3_te), tr(X4_te)

    train_loader = DataLoader(QuadDataset(X1_tr, X2_tr, X3_tr, X4_tr),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(QuadDataset(X1_va, X2_va, X3_va, X4_va),
                              batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(QuadDataset(X1_te, X2_te, X3_te, X4_te),
                              batch_size=batch_size, shuffle=False)

    torch.manual_seed(random_state + fold_id)

    model = RewardMoECosineSplit(
        in_dim_full=F_DIM,
        base_idx=base_idx,
        aux_idx=aux_idx,
        participant_idx=participant_idx,
        embed_dim=32,
        trunk_hidden=hidden,
        expert_hidden=(128, 64, 32),
        dropout=dropout,
        learn_scale=True,
    ).to(device)

    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs, eta_min=1e-6)

    # --- best tracking ---
    if monitor == "val_loss":
        mode = "min"
        best_score = float("inf")
    elif monitor == "val_top1_acc":
        mode = "max"
        best_score = -float("inf")
    else:
        raise ValueError("monitor must be 'val_loss' or 'val_top1_acc'")

    best_epoch = 0
    best_state = None
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for x1, x2, x3, x4 in train_loader:
            x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

            s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)
            loss = loss_listwise_top1(s1, s2, s3, s4, temperature=temperature)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        scheduler.step()

        trm = eval_metrics(model, train_loader, device, temperature=temperature)
        vam = eval_metrics(model, val_loader, device, temperature=temperature)

        # мониторим
        current = vam["loss"] if monitor == "val_loss" else vam["top1_acc_soft"]
        improved = (current < best_score - min_delta) if mode == "min" else (current > best_score + min_delta)

        if improved:
            best_score = float(current)
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch == 1 or epoch % 25 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:03d} | "
                f"train loss {trm['loss']:.4f} top1 {trm['top1_acc_soft']:.3f} | "
                f"val loss {vam['loss']:.4f} top1 {vam['top1_acc_soft']:.3f}"
            )

    if restore_best and best_state is not None:
        model.load_state_dict(best_state)

    # --- финальные метрики ---
    val_metrics  = eval_metrics(model, val_loader, device, temperature=temperature)
    test_metrics = eval_metrics(model, test_loader, device, temperature=temperature)

    info = {
        "best_epoch": best_epoch,
        "best_score": best_score,
        "monitor": monitor,
        "sizes": {"train": len(df_train_fold), "val": len(df_val_fold), "test": len(df_test_holdout)},
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }

    return model, scaler, device, info


def kfold_cv_ranknet_quads(
    df: pd.DataFrame,
    *,
    k=5,
    test_size=0.2,          # общий holdout TEST от всего df
    random_state=0,
    shuffle=True,
    hidden=(64, 32),
    dropout=0.1,
    temperature=1.0,
    batch_size=64,
    epochs=200,
    lr=5e-4,
    weight_decay=1e-3,
    device=None,
    monitor="val_loss",
    min_delta=0.0,
    restore_best=True,
):
    # --- 1) общий TEST holdout ---
    idx = np.arange(len(df))
    idx_trainval, idx_test = train_test_split(
        idx, test_size=test_size, random_state=random_state, shuffle=True
    )

    df_test = df.iloc[idx_test].reset_index(drop=True)
    df_trainval = df.iloc[idx_trainval].reset_index(drop=True)

    # индексы внутри trainval
    idx_tv = np.arange(len(df_trainval))

    kf = KFold(n_splits=k, shuffle=shuffle, random_state=random_state)

    fold_rows = []
    fold_models = []
    fold_scalers = []
    fold_splits = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(idx_tv), start=1):
        print(f"\n=== Fold {fold}/{k} | train={len(tr_idx)} val={len(va_idx)} | TEST holdout={len(df_test)} ===")

        df_tr = df_trainval.iloc[tr_idx].reset_index(drop=True)
        df_va = df_trainval.iloc[va_idx].reset_index(drop=True)

        model, scaler, device_used, info = train_one_fold_quads(
            df_train_fold=df_tr,
            df_val_fold=df_va,
            df_test_holdout=df_test,
            hidden=hidden,
            dropout=dropout,
            temperature=temperature,
            batch_size=batch_size,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            device=device,
            monitor=monitor,
            min_delta=min_delta,
            restore_best=restore_best,
            fold_id=fold,
        )

        row = {
            "fold": fold,
            "best_epoch": info["best_epoch"],
            "best_score": info["best_score"],
            "monitor": info["monitor"],
            "n_train": info["sizes"]["train"],
            "n_val": info["sizes"]["val"],
            "n_test": info["sizes"]["test"],
            # val:
            "val_loss": info["val_metrics"]["loss"],
            "val_top1": info["val_metrics"]["top1_acc_soft"],
            "val_margin": info["val_metrics"]["mean_margin"],
            # test:
            "test_loss": info["test_metrics"]["loss"],
            "test_top1": info["test_metrics"]["top1_acc_soft"],
            "test_margin": info["test_metrics"]["mean_margin"],
        }
        fold_rows.append(row)

        fold_models.append(model)
        fold_scalers.append(scaler)
        fold_splits.append({"df_train": df_tr, "df_val": df_va, "df_test": df_test})

        print(f"Fold {fold} | val_top1={row['val_top1']:.3f} test_top1={row['test_top1']:.3f} best epoch={row['best_epoch']}")

    results = pd.DataFrame(fold_rows)

    summary = {
        "val_top1_mean": float(results["val_top1"].mean()),
        "val_top1_std": float(results["val_top1"].std(ddof=1)) if k > 1 else 0.0,
        "test_top1_mean": float(results["test_top1"].mean()),
        "test_top1_std": float(results["test_top1"].std(ddof=1)) if k > 1 else 0.0,
        "val_loss_mean": float(results["val_loss"].mean()),
        "test_loss_mean": float(results["test_loss"].mean()),
    }

    print("\n=== CV Summary (with fixed TEST holdout) ===")
    print(f"VAL  top1:  mean={summary['val_top1_mean']:.3f} std={summary['val_top1_std']:.3f}")
    print(f"TEST top1:  mean={summary['test_top1_mean']:.3f} std={summary['test_top1_std']:.3f}")
    print(f"VAL  loss:  mean={summary['val_loss_mean']:.4f}")
    print(f"TEST loss:  mean={summary['test_loss_mean']:.4f}")

    info = {
        "idx": {"trainval": idx_trainval, "test": idx_test},
        "df_test": df_test,
        "results": results,
        "summary": summary,
        "models": fold_models,
        "scalers": fold_scalers,
        "splits": fold_splits,
    }
    return info

In [ ]:
temp = 1.1
cv = kfold_cv_ranknet_quads(
    df,
    k=4,
    hidden=(128, 64),
    dropout=0.5,
    temperature=temp,
    batch_size=256,
    epochs=200,
    lr=9e-4,
    weight_decay=1e-3,
    monitor="val_loss",
)

cv["results"]   # таблица по фолдам
cv["summary"]   # среднее/стд


=== Fold 1/4 | train=2805 val=935 | TEST holdout=935 ===
epoch 001 | train loss 1.2701 top1 0.508 | val loss 1.2794 top1 0.486
epoch 025 | train loss 1.0143 top1 0.601 | val loss 1.0595 top1 0.578
epoch 050 | train loss 0.9552 top1 0.638 | val loss 1.0322 top1 0.598
epoch 075 | train loss 0.9198 top1 0.655 | val loss 1.0241 top1 0.594
epoch 100 | train loss 0.8952 top1 0.664 | val loss 1.0183 top1 0.584
epoch 125 | train loss 0.8776 top1 0.675 | val loss 1.0174 top1 0.590
epoch 150 | train loss 0.8683 top1 0.677 | val loss 1.0181 top1 0.591
epoch 175 | train loss 0.8652 top1 0.678 | val loss 1.0170 top1 0.586
epoch 200 | train loss 0.8643 top1 0.679 | val loss 1.0175 top1 0.586
Fold 1 | val_top1=0.591 test_top1=0.560 best epoch=131

=== Fold 2/4 | train=2805 val=935 | TEST holdout=935 ===
epoch 001 | train loss 1.2829 top1 0.486 | val loss 1.2793 top1 0.474
epoch 025 | train loss 1.0193 top1 0.591 | val loss 1.0325 top1 0.567
epoch 050 | train loss 0.9683 top1 0.622 | val loss 1.0033 

{'val_top1_mean': 0.6010695187165775,
 'val_top1_std': 0.01111476454054892,
 'test_top1_mean': 0.567379679144385,
 'test_top1_std': 0.008121772370691269,
 'val_loss_mean': 0.9944989181616727,
 'test_loss_mean': 1.036243813943098}

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
df_test = cv["df_test"]

reports = []

for i, (model, scaler) in enumerate(zip(cv["models"], cv["scalers"]), start=1):
    r = eval_test_all_participants(
        df_test,
        model,
        scaler,
        device,
        temperature=temp,
    )
    r["fold"] = i
    reports.append(r)

reports_df = pd.concat(reports, ignore_index=False)
for i in range(len(reports)):
  print(reports[i])

  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  347  1.158896   0.512968   0.512968       0.0    -0.058948   
2           Y  305  0.843440   0.649180   0.649180       0.0     0.433154   
1           F  283  1.102324   0.522968   0.522968       0.0     0.017407   

   pwin_mean  pwin_q05  pwin_q95  fold  
0   0.385680  0.077444  0.699530     1  
2   0.487260  0.111365  0.767052     1  
1   0.386348  0.093849  0.663798     1  
  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  347  1.143298   0.521614   0.521614       0.0    -0.046455   
2           Y  305  0.840660   0.649180   0.649180       0.0     0.442984   
1           F  283  1.127447   0.537102   0.537102       0.0    -0.042115   

   pwin_mean  pwin_q05  pwin_q95  fold  
0   0.391587  0.083479  0.699716     2  
2   0.493428  0.109689  0.783581     2  
1   0.375904  0.096714  0.663446     2  
  participant    n      loss  top1_soft  top1_hard  ti

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


def _summary_stats_1d(t: torch.Tensor) -> dict:
    """t: (N,) on any device"""
    t = t.detach()
    q05, q50, q95 = torch.quantile(t, torch.tensor([0.05, 0.50, 0.95], device=t.device))
    return {
        "mean": float(t.mean().item()),
        "std": float(t.std(unbiased=False).item()),
        "q05": float(q05.item()),
        "q50": float(q50.item()),
        "q95": float(q95.item()),
        "min": float(t.min().item()),
        "max": float(t.max().item()),
    }


@torch.no_grad()
def ensemble_eval_metrics_quads(
    df,
    models,
    scalers,
    device,
    *,
    temperature: float = 1.0,
    batch_size: int = 256,
    tie_eps: float = 1e-6,
):
    """
    Ensemble метрики для четверок:
      - усредняем скоры s1..s4 по моделям (каждая со своим scaler)
      - считаем listwise CE loss (winner = x1)
      - считаем top1_hard, top1_soft, tie_rate, margin, pwin summary

    Возвращает словарь в стиле твоего eval_metrics.
    """
    assert len(models) == len(scalers), "models and scalers must have same length"
    m = len(models)
    if m == 0:
        raise ValueError("Empty models list")

    # raw arrays
    X1, X2, X3, X4 = prepare_arrays(df)

    # loader по raw (без нормализации)
    ds = QuadDataset(
        X1.astype(np.float32),
        X2.astype(np.float32),
        X3.astype(np.float32),
        X4.astype(np.float32),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    total = 0
    total_loss = 0.0
    top1_correct = 0
    top1_soft_sum = 0.0
    tie_count = 0
    margin_sum = 0.0

    pwin_all = []

    for x1, x2, x3, x4 in loader:
        bs = x1.size(0)

        # усредняемые скоры на device
        s1_mean = torch.zeros(bs, device=device)
        s2_mean = torch.zeros(bs, device=device)
        s3_mean = torch.zeros(bs, device=device)
        s4_mean = torch.zeros(bs, device=device)

        # CPU -> numpy для scaler
        x1_np = x1.numpy()
        x2_np = x2.numpy()
        x3_np = x3.numpy()
        x4_np = x4.numpy()

        for model, scaler in zip(models, scalers):
            model.eval()

            x1n = torch.from_numpy(scaler.transform(x1_np).astype(np.float32)).to(device)
            x2n = torch.from_numpy(scaler.transform(x2_np).astype(np.float32)).to(device)
            x3n = torch.from_numpy(scaler.transform(x3_np).astype(np.float32)).to(device)
            x4n = torch.from_numpy(scaler.transform(x4_np).astype(np.float32)).to(device)

            s1_mean += model(x1n)
            s2_mean += model(x2n)
            s3_mean += model(x3n)
            s4_mean += model(x4n)

        s1_mean /= m
        s2_mean /= m
        s3_mean /= m
        s4_mean /= m

        # logits для listwise loss / pwin
        logits = torch.stack([s1_mean, s2_mean, s3_mean, s4_mean], dim=1) / temperature  # (B,4)
        target = torch.zeros(bs, dtype=torch.long, device=device)

        loss = F.cross_entropy(logits, target)
        total_loss += float(loss.item()) * bs
        total += bs

        # hard top1
        pred = logits.argmax(dim=1)
        top1_correct += int((pred == 0).sum().item())

        # margin
        best_loser = torch.max(torch.stack([s2_mean, s3_mean, s4_mean], dim=1), dim=1).values
        margin = (s1_mean - best_loser)  # (B,)
        margin_sum += float(margin.mean().item()) * bs

        # tie: если margin очень мал (победитель ≈ лучший проигравший)
        tie_count += int((margin.abs() < tie_eps).sum().item())

        # top1_soft: “мягкая уверенность” победы через sigmoid(margin)
        # (это обычно ближе к твоим top1_soft, чем pwin=softmax)
        top1_soft_sum += float(torch.sigmoid(margin).mean().item()) * bs

        # p(win): softmax вероятность winner среди 4
        pwin = torch.softmax(logits, dim=1)[:, 0]  # (B,)
        pwin_all.append(pwin.detach().cpu())

    pwin_all = torch.cat(pwin_all, dim=0) if len(pwin_all) else torch.empty(0)

    return {
        "loss": total_loss / max(total, 1),
        "top1_acc_soft": top1_soft_sum / max(total, 1),
        "top1_acc_hard": top1_correct / max(total, 1),
        "tie_rate": tie_count / max(total, 1),
        "mean_margin": margin_sum / max(total, 1),
        "pwin": _summary_stats_1d(pwin_all) if pwin_all.numel() else None,
        "n": total,
    }

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
df_part = cv["df_test"]
df_part = df_part[df_part['x1_participant_Y']==1]
ens_test = ensemble_eval_metrics_quads(
    df_part,
    cv["models"],
    cv["scalers"],
    device,
    temperature=temp,
    batch_size=256,
)

print("[ENSEMBLE TEST]", ens_test)

[ENSEMBLE TEST] {'loss': 0.8348014096744725, 'top1_acc_soft': 0.5843164309126432, 'top1_acc_hard': 0.6557377049180327, 'tie_rate': 0.0, 'mean_margin': 0.4229529872292378, 'pwin': {'mean': 0.4892547130584717, 'std': 0.19362737238407135, 'q05': 0.11668076366186142, 'q50': 0.5264773964881897, 'q95': 0.7688121795654297, 'min': 0.05006171762943268, 'max': 0.800624668598175}, 'n': 305}
